## Explore data

This will help with the development of the preprocessing pipeline

In [1]:
import pandas as pd
from pathlib import Path
import re
import numpy as np
from collections import Counter

RAW_DATA_PATH = Path('..') / 'Data' / 'raw'

## UCI Phishing URL dataset

In [2]:
file_path = RAW_DATA_PATH / 'phishing_url_uci.pkl'

df_uci = pd.read_pickle(file_path)

In [3]:
df_uci.head()

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1


In [4]:
df_uci.columns

Index(['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP',
       'TLD', 'URLSimilarityIndex', 'CharContinuationRate',
       'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain',
       'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio',
       'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL',
       'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL',
       'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL',
       'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength',
       'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore',
       'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect',
       'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame',
       'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton',
       'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto',
       'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef',
       'NoOfEmptyRef', 'NoOf

### Kaggle Datasets

kaggle phishing datasets have 2 types of data:
1. Phishing datas including URLs data
2. Top 1m websites data ( for negative samples )

#### Dataset type 1: Phishing datasets including URLs data

In [5]:
file_path = RAW_DATA_PATH / 'phishing_site_urls.csv'

df_kaggle_urls = pd.read_csv(file_path)

In [6]:
df_kaggle_urls.head()

,URL,Label
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,bad
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,bad
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,bad
3,mail.printakid.com/www.online.americanexpress....,bad
4,thewhiskeydregs.com/wp-content/themes/widescre...,bad


In [7]:
df_kaggle_urls.columns

Index(['URL', 'Label'], dtype='str')

#### Dataset type 3: Top 1M websites data

Possibly don't need to use this dataset, but it can be useful for negative samples and for understanding the distribution of legitimate URLs.

In [16]:
file_path = RAW_DATA_PATH /'top-1m.csv'

df_kaggle_top1m = pd.read_csv(file_path)

In [17]:
df_kaggle_top1m.head()

,rank,url
0,1,google.com
1,2,youtube.com
2,3,facebook.com
3,4,baidu.com
4,5,wikipedia.org


In [18]:
df_kaggle_top1m.columns

Index(['rank', 'url'], dtype='str')

### Transformation

In [11]:
df_kaggle_urls.head()

,URL,Label
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,bad
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,bad
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,bad
3,mail.printakid.com/www.online.americanexpress....,bad
4,thewhiskeydregs.com/wp-content/themes/widescre...,bad


In [12]:
df_kaggle_urls['URLLength'] = df_kaggle_urls['URL'].apply(lambda x: len(x))

df_kaggle_urls['Domain'] = df_kaggle_urls['URL'].apply(lambda x: x.split('/')[2] if len(x.split('/')) > 2 else '')

df_kaggle_urls['IsHTTPS'] = df_kaggle_urls['URL'].apply(lambda x: 1 if x.startswith('https://') else 0)

df_kaggle_urls['HasTitle'] = df_kaggle_urls['URL'].apply(lambda x: 1 if '<title>' in x.lower() else 0)

df_kaggle_urls['HasFavicon'] = df_kaggle_urls['URL'].apply(lambda x: 1 if 'favicon' in x.lower() else 0)

df_kaggle_urls['DomainLength'] = df_kaggle_urls['Domain'].apply(lambda x: len(x))

df_kaggle_urls['IsDomainIP'] = df_kaggle_urls['Domain'].apply(lambda x: 1 if all(part.isdigit() for part in x.split('.')) else 0)

df_kaggle_urls['NumSubdomains'] = df_kaggle_urls['Domain'].apply(lambda x: len(x.split('.')) - 1)

df_kaggle_urls['TLD'] = df_kaggle_urls['Domain'].apply(lambda x: x.split('.')[-1] if len(x.split('.')) > 1 else '')

df_kaggle_urls['TLDLength'] = df_kaggle_urls['TLD'].apply(lambda x: len(x))


df_kaggle_urls['NoOfSubDomain'] = df_kaggle_urls['Domain'].apply(
    lambda d: max(len(d.split('.')) - 2, 0) if d else 0
)

url_s = df_kaggle_urls['URL'].astype(str)

df_kaggle_urls['NoOfLettersInURL'] = url_s.str.count(r'[A-Za-z]')
df_kaggle_urls['NoOfDegitsInURL'] = url_s.str.count(r'[0-9]')

df_kaggle_urls['LetterRatioInURL'] = (
    df_kaggle_urls['NoOfLettersInURL'] / df_kaggle_urls['URLLength']
).fillna(0)

df_kaggle_urls['DegitRatioInURL'] = (
    df_kaggle_urls['NoOfDegitsInURL'] / df_kaggle_urls['URLLength']
).fillna(0)

df_kaggle_urls['NoOfEqualsInURL'] = url_s.str.count(r'=')
df_kaggle_urls['NoOfQMarkInURL'] = url_s.str.count(r'\?')
df_kaggle_urls['NoOfAmpersandInURL'] = url_s.str.count(r'&')

total_special = url_s.str.count(r'[^A-Za-z0-9]')

df_kaggle_urls['NoOfOtherSpecialCharsInURL'] = (
    total_special
    - df_kaggle_urls['NoOfEqualsInURL']
    - df_kaggle_urls['NoOfQMarkInURL']
    - df_kaggle_urls['NoOfAmpersandInURL']
).clip(lower=0)

df_kaggle_urls['SpacialCharRatioInURL'] = (
    (df_kaggle_urls['NoOfEqualsInURL']
     + df_kaggle_urls['NoOfQMarkInURL']
     + df_kaggle_urls['NoOfAmpersandInURL']
     + df_kaggle_urls['NoOfOtherSpecialCharsInURL'])
    / df_kaggle_urls['URLLength']
).fillna(0)


df_kaggle_urls['NoOfObfuscatedChar'] = url_s.str.count(r'[%@]')
df_kaggle_urls['HasObfuscation'] = (df_kaggle_urls['NoOfObfuscatedChar'] > 0).astype(int)
df_kaggle_urls['ObfuscationRatio'] = (
    df_kaggle_urls['NoOfObfuscatedChar'] / df_kaggle_urls['URLLength']
).fillna(0)


lower_url = url_s.str.lower()
df_kaggle_urls['Bank'] = lower_url.str.contains('bank').astype(int)
df_kaggle_urls['Pay'] = lower_url.str.contains('pay').astype(int)
df_kaggle_urls['Crypto'] = lower_url.str.contains('crypto|bitcoin|btc|eth').astype(int)


def char_continuation_rate(u: str) -> float:
    s = str(u)
    if len(s) < 2:
        return 0.0

    def cat(ch: str) -> str:
        if ch.isalpha():
            return 'A'
        if ch.isdigit():
            return 'D'
        return 'S'  # specials

    prev = cat(s[0])
    same, total = 0, 0
    for ch in s[1:]:
        c = cat(ch)
        if c == prev:
            same += 1
        total += 1
        prev = c

    return same / total if total else 0.0

df_kaggle_urls['CharContinuationRate'] = df_kaggle_urls['URL'].apply(char_continuation_rate)

df_kaggle_urls['label_binary'] = df_kaggle_urls['Label'].apply(lambda x: 1 if x == 'good' else 0)

tld_legit_prob = df_kaggle_urls.groupby('TLD')['label_binary'].mean()
default_tld_prob = df_kaggle_urls['label_binary'].mean()

df_kaggle_urls['TLDLegitimateProb'] = (
    df_kaggle_urls['TLD']
        .map(tld_legit_prob)
        .fillna(default_tld_prob)
)

urls_legit = df_kaggle_urls[df_kaggle_urls['label_binary'] == 1]

all_chars = ''.join(df_kaggle_urls['URL'].astype(str).tolist())
char_counts = Counter(all_chars)
total_chars = sum(char_counts.values())

# simple Laplace smoothing over printable ASCII
alphabet = [chr(i) for i in range(32, 127)]
alpha_size = len(alphabet)
denom = total_chars + alpha_size

char_probs = {ch: (char_counts.get(ch, 0) + 1) / denom for ch in alphabet}

def url_char_prob(u: str) -> float:
    s = str(u)
    if not s:
        return 0.0
    log_p = 0.0
    for ch in s:
        p = char_probs.get(ch, 1 / denom)
        log_p += np.log(p)
    avg_log = log_p / len(s)
    # back to (0,1]: geometric mean probability per char
    return float(np.exp(avg_log))

df_kaggle_urls['URLCharProb'] = df_kaggle_urls['URL'].astype(str).apply(url_char_prob)

df_kaggle_urls['URLSimilarityIndex'] = df_kaggle_urls['URLCharProb'] * 100.0

In [13]:
df_kaggle_urls.head()

,URL,Label,URLLength,Domain,IsHTTPS,HasTitle,HasFavicon,DomainLength,IsDomainIP,NumSubdomains,...,HasObfuscation,ObfuscationRatio,Bank,Pay,Crypto,CharContinuationRate,label_binary,TLDLegitimateProb,URLCharProb,URLSimilarityIndex
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,bad,225,login.SkyPe.com,0,0,0,15,0,2,...,0,0.0,0,0,0,0.633929,0,0.067990,0.022210,2.220987
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,bad,81,cycgi-bin,0,0,0,9,0,0,...,0,0.0,0,1,0,0.650000,0,0.734560,0.028614,2.861394
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,bad,177,href,0,0,0,4,0,0,...,0,0.0,0,1,0,0.659091,0,0.734560,0.025362,2.536212
3,mail.printakid.com/www.online.americanexpress....,bad,60,index.html,0,0,0,10,0,1,...,0,0.0,0,0,0,0.728814,0,0.823275,0.038129,3.812929
4,thewhiskeydregs.com/wp-content/themes/widescre...,bad,116,themes,0,0,0,6,0,0,...,0,0.0,0,0,0,0.808696,0,0.734560,0.021561,2.156097


In [14]:
df_uci.head()

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1
